# evalbridge quickstart

Most ML teams solve the same problems from scratch, every sprint.

**Gap 1** — no standard evaluation format. Every notebook reimplements accuracy and AUC differently.

**Gap 2** — offline evaluation and live A/B testing are two separate codebases. You evaluate in Jupyter, then rewrite everything for production.

**Gap 3** — drift is detected but nothing acts on it automatically. A human still rolls back the model manually.

**evalbridge fixes all three.**

In [ ]:
# Uncomment to install:
# !pip install evalbridge scikit-learn -q

## Generate a synthetic dataset

We train two classifiers on the same data — a `LogisticRegression` baseline and a `RandomForestClassifier` challenger. The challenger will be slightly better.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10, n_redundant=5,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train, y_train)

challenger_model = RandomForestClassifier(n_estimators=100, random_state=42)
challenger_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict_proba(X_test)[:, 1]
challenger_preds = challenger_model.predict_proba(X_test)[:, 1]

# Show the gap
from sklearn.metrics import roc_auc_score
print(f"Baseline  AUC: {roc_auc_score(y_test, baseline_preds):.4f}")
print(f"Challenger AUC: {roc_auc_score(y_test, challenger_preds):.4f}")

Baseline  AUC: 0.9207
Challenger AUC: 0.9703


## Gap 1 — standard evaluation format

One `Experiment`, one `evaluate()` call. All metrics computed the same way, every time.

In [2]:
from evalbridge import Experiment

exp = Experiment("churn_v2_vs_v3", min_samples=100, min_confidence=0.90)
exp.log("baseline",   y_true=y_test.tolist(), y_pred=baseline_preds.tolist())
exp.log("challenger", y_true=y_test.tolist(), y_pred=challenger_preds.tolist())

result = exp.evaluate()
result.summary()


┌──────────────┬──────────┬──────────┬────────┬──────────┬─────────────┬────────────┬───┐
│ Model        │ Accuracy │ AUC-ROC  │   F1   │ Log-loss │ Brier score │  Samples   │   │
├──────────────┼──────────┼──────────┼────────┼──────────┼─────────────┼────────────┼───┤
│ baseline     │    0.856 │    0.921 │  0.855 │    0.366 │       0.110 │        800 │    │
│ challenger   │    0.914 │    0.970 │  0.915 │    0.294 │       0.080 │        800 │ ◀ │
└──────────────┴──────────┴──────────┴────────┴──────────┴─────────────┴────────────┴───┘
  Confidence  [████████████████████] 100.0%
  Status      READY ✓
  Winner      challenger



In [3]:
print(f"Winner:     {result.winner}")
print(f"Confidence: {result.confidence:.4f}")
print(f"Ready:      {result.ready}")
print(f"\nBaseline metrics:  {result.metrics['baseline']}")
print(f"Challenger metrics: {result.metrics['challenger']}")

Winner:     challenger
Confidence: 0.9998
Ready:      True

Baseline metrics:  {'accuracy': 0.85625, 'auc_roc': 0.9206875, 'f1': 0.854614412136536, 'log_loss': 0.36632645783059237, 'brier_score': 0.11043899681517942, 'n_samples': 800}
Challenger metrics: {'accuracy': 0.91375, 'auc_roc': 0.970259375, 'f1': 0.9153374233128835, 'log_loss': 0.2937381525303484, 'brier_score': 0.079875, 'n_samples': 800}


## Gap 2 — offline → live, same object

Save the experiment from the notebook. Load it in production. Continue logging live data.
No rewriting. The same `Experiment` object works in both contexts.

In [4]:
import evalbridge

# Save state (would normally be done at end of notebook)
exp.save("/tmp/churn_v2_vs_v3.evalbridge")
print("Saved experiment to /tmp/churn_v2_vs_v3.evalbridge")

# Load in 'production'
exp_live = Experiment.load("/tmp/churn_v2_vs_v3.evalbridge")
exp_live.go_live(traffic_split=0.1)

# Register so @evalbridge.track can find it
evalbridge.register_experiment(exp_live)

Saved experiment to /tmp/churn_v2_vs_v3.evalbridge
[evalbridge] 'churn_v2_vs_v3' is now live — 10% traffic to challenger


In [5]:
# Simulate 200 live requests using the @evalbridge.track decorator
import random
random.seed(0)

@evalbridge.track(experiment="churn_v2_vs_v3", model="challenger")
def predict_challenger(features):
    idx = random.randint(0, len(X_test) - 1)
    return float(challenger_model.predict_proba([features])[0, 1])

@evalbridge.track(experiment="churn_v2_vs_v3", model="baseline")
def predict_baseline(features):
    idx = random.randint(0, len(X_test) - 1)
    return float(baseline_model.predict_proba([features])[0, 1])

router = exp_live._router
request_ids = []
ground_truths = []

for i in range(200):
    idx = i % len(X_test)
    features = X_test[idx]
    model_to_use = router.route()
    if model_to_use == "challenger":
        predict_challenger(features)
        rid = predict_challenger.last_request_id
    else:
        predict_baseline(features)
        rid = predict_baseline.last_request_id
    request_ids.append(rid)
    ground_truths.append(int(y_test[idx]))

# Attach ground truth
for rid, gt in zip(request_ids, ground_truths):
    evalbridge.log_outcome(request_id=rid, y_true=gt)

print(f"Live data accumulated: {len(exp_live._data['baseline']['y_true'])} baseline, "
      f"{len(exp_live._data['challenger']['y_true'])} challenger")

Live data accumulated: 980 baseline, 820 challenger


In [6]:
# Evaluate again — now includes both offline and live data
result_live = exp_live.evaluate()
result_live.summary()


┌──────────────┬──────────┬──────────┬────────┬──────────┬─────────────┬────────────┬───┐
│ Model        │ Accuracy │ AUC-ROC  │   F1   │ Log-loss │ Brier score │  Samples   │   │
├──────────────┼──────────┼──────────┼────────┼──────────┼─────────────┼────────────┼───┤
│ baseline     │    0.864 │    0.929 │  0.861 │    0.347 │       0.105 │        980 │    │
│ challenger   │    0.913 │    0.971 │  0.915 │    0.293 │       0.079 │        820 │ ◀ │
└──────────────┴──────────┴──────────┴────────┴──────────┴─────────────┴────────────┴───┘
  Confidence  [████████████████████] 100.0%
  Status      READY ✓
  Winner      challenger



## Gap 3 — drift detection and auto-rollback

Detect when the production distribution shifts, and automatically take action.

In [7]:
from evalbridge import DriftDetector
import numpy as np

# Reference distribution = baseline predictions
detector = DriftDetector(reference=baseline_preds, threshold=0.2)

# Simulate stable production (same distribution)
stable_preds = baseline_preds + np.random.default_rng(0).normal(0, 0.01, len(baseline_preds))
stable_preds = np.clip(stable_preds, 0, 1)
stable_report = detector.check(stable_preds)
print(f"Stable: PSI={stable_report.psi:.4f}, severity={stable_report.severity}, alert={stable_report.alert}")

# Simulate drifted production (shifted distribution)
drifted_preds = np.random.default_rng(1).beta(0.5, 5, len(baseline_preds))
drift_report = detector.check(drifted_preds)
print(f"Drifted: PSI={drift_report.psi:.4f}, severity={drift_report.severity}, alert={drift_report.alert}")
print(f"Summary: {drift_report.summary}")

Stable: PSI=0.0039, severity=none, alert=False
Drifted: PSI=7.6998, severity=severe, alert=True
Summary: Severe drift detected (PSI=7.6998) — investigate


In [8]:
# Auto-rollback when drift threshold is crossed
rollback_fired = []

exp_drift = Experiment("churn_drift_demo", min_samples=50)
exp_drift.log("baseline",   y_true=y_test.tolist(), y_pred=baseline_preds.tolist())
exp_drift.log("challenger", y_true=y_test.tolist(), y_pred=challenger_preds.tolist())

exp_drift.on_drift(threshold=0.0, action=lambda r: rollback_fired.append(True))

# Inject drifted challenger data
exp_drift.log("challenger", y_true=[0]*200, y_pred=drifted_preds[:200].tolist())
exp_drift.evaluate()  # triggers drift check internally

print(f"Rollback fired: {len(rollback_fired) > 0}")

Rollback fired: True


## Bandit — auto traffic shifting

Thompson sampling learns which model performs better and shifts traffic automatically.

In [9]:
exp_bandit = Experiment("churn_bandit", min_samples=10, method="bandit")
exp_bandit.log("baseline",   y_true=y_test[:100].tolist(), y_pred=baseline_preds[:100].tolist())
exp_bandit.log("challenger", y_true=y_test[:100].tolist(), y_pred=challenger_preds[:100].tolist())

bandit = exp_bandit.as_bandit(strategy="thompson")

rng = np.random.default_rng(42)

# Challenger wins 80% of the time, baseline 65%
for _ in range(500):
    arm = bandit.route()
    reward = float(rng.random() < (0.80 if arm == "challenger" else 0.65))
    bandit.update(arm, reward)

allocs = bandit.allocations()
print(f"Final allocations after 500 rounds:")
print(f"  baseline   : {allocs['baseline']*100:.1f}%")
print(f"  challenger : {allocs['challenger']*100:.1f}%")
bandit.summary()

Final allocations after 500 rounds:
  baseline   : 4.8%
  challenger : 95.2%
Strategy: thompson
  baseline   :   4.8%
  challenger :  95.2%
  P(challenger > baseline): 0.649


'Strategy: thompson\n  baseline   :   4.8%\n  challenger :  95.2%\n  P(challenger > baseline): 0.649'

## What's next

- **GitHub**: https://github.com/YOUR_USERNAME/evalbridge
- **PyPI**: `pip install evalbridge`
- **Integrations**: `pip install evalbridge[mlflow]`, `evalbridge[wandb]`, `evalbridge[hf]`
- **CLI**: `evalbridge report experiment.evalbridge` to generate an HTML report from any saved experiment